# Suite CMP — Comparison

`Comparison.OfX` implements `Visitors.OfX` and records what is written into it; `a.compare(b)` returns -1, 0 or 1,
or `null` when the two are incomparable. Equality follows EQUALITY.md: natives by canonical value,
objects by their declared properties, links by identity, adjacency entries as sets. Ordering is defined only for
`int`, `float`, `str` and `bytes`.

In [ ]:
import { Comparison as C, Proxies, Schemas as S } from "@mbse/schemas/Framework";
import { AttributeError, KeyError, NotImplementedError, ValueError } from "@mbse/schemas/Framework/Errors";
import type { Native, Visitable } from "@mbse/schemas/Framework/Visitors";
import { assert, get_property, raises, same, set_property, text } from "./support.js";

const store = new Proxies.OfStore();

const [Int, Float, Str, Bool, Bytes] = [BigInt, Number, String, Boolean, Uint8Array].map((t) => new S.OfNative.Data(t));
const bytes = (...values: number[]) => new Uint8Array(values);

/** Compares two native values under `schema`. */
function cmp(schema: S.OfNative.Data, a: Native, b: Native): C.Result {
  return new C.OfNative(schema, a).compare(new C.OfNative(schema, b));
}

const Tag = new S.OfRelation.Builder().links("item", "tag").properties(text("note")).create();
const Point = new S.OfObject.Builder().ref().properties(text("x", BigInt), text("label")).relations(
  (r) => r.name("tags").of(Tag).me("item")).create();
const Label = new S.OfObject.Builder().ref().properties(text("name")).relations((r) => r.name("items").of(Tag).me("tag")).create();
for (const [name, schema] of [["Point", Point], ["Label", Label], ["Tag", Tag]] as const) store.register(name, schema);
const B = store;

## CMP-01 · Ordered natives: ints, strings by code point, bytes lexicographically

In [ ]:
assert(cmp(Int, 1n, 2n) === -1 && cmp(Int, 2n, 1n) === 1 && cmp(Int, 2n, 2n) === 0);
assert(cmp(Int, 2n ** 70n, 2n ** 70n + 1n) === -1); // beyond 2**53
assert(cmp(Str, "a", "b") === -1 && cmp(Str, "ab", "a") === 1 && cmp(Str, "", "") === 0);
assert(cmp(Str, "\uffff", "\u{10000}") === -1); // by code point, not by UTF-16 code unit
assert(cmp(Str, "e\u0301", "\u00e9") === -1); // no Unicode normalization
assert(cmp(Bytes, bytes(1), bytes(2)) === -1 && cmp(Bytes, bytes(1, 0), bytes(1)) === 1 && cmp(Bytes, bytes(), bytes()) === 0);
assert(cmp(Bytes, bytes(0xff), bytes(1, 0)) === 1);

## CMP-02 · Floats: by value, `-0.0` before `0.0`, NaNs equal each other and nothing else

In [ ]:
assert(cmp(Float, 1.5, 2.5) === -1 && cmp(Float, -Infinity, Infinity) === -1 && cmp(Float, 0.1 + 0.2, 0.3) === 1);
assert(cmp(Float, -0, 0) === -1 && cmp(Float, 0, -0) === 1 && cmp(Float, -0, -0) === 0);
assert(cmp(Float, NaN, NaN) === 0 && cmp(Float, NaN, -NaN) === 0);
assert(cmp(Float, NaN, 1.0) === null && cmp(Float, Infinity, NaN) === null);

## CMP-03 · Booleans are equal or incomparable

In [ ]:
assert(cmp(Bool, true, true) === 0 && cmp(Bool, false, false) === 0);
assert(cmp(Bool, false, true) === null && cmp(Bool, true, false) === null);

## CMP-04 · Absent values, wrong types and distinct native types

In [ ]:
const absent = new C.OfNative(Int);
assert(!absent.has() && absent.compare(new C.OfNative(Int)) === 0);
assert(absent.compare(new C.OfNative(Int, 1n)) === null && new C.OfNative(Int, 1n).compare(absent) === null);
raises(AttributeError, () => absent.get(), "the value is not set");
const value = new C.OfNative(Int, 1n);
assert(value.get() === 1n && value.clear() === value && !value.has());
assert(value.set(3n) === value && value.get() === 3n);
raises(TypeError, () => new C.OfNative(Int, true), "expected int, got bool");
raises(TypeError, () => new C.OfNative(Float).set(1n), "expected float, got int");
assert(new C.OfNative(Int, 1n).compare(new C.OfNative(Float, 1.0)) === null); // distinct native types are never equal
assert(new C.OfNative(new S.OfNative.Data(BigInt), 1n).compare(new C.OfNative(Int, 1n)) === 0); // the same native type

## CMP-05 · Objects are equal when their declared properties are, and are otherwise incomparable

In [ ]:
const p = B.Point().x(1n).label("a").create();
const unlabeled = B.Point().x(1n).create();
assert(new C.OfObject(Point, p).compare(new C.OfObject(Point, B.Point(p).clone())) === 0);
assert(new C.OfObject(Point, p).compare(new C.OfObject(Point, B.Point().x(2n).label("a").create())) === null); // not ordered
assert(new C.OfObject(Point, p).compare(new C.OfObject(Point, unlabeled)) === null); // present and absent
assert(new C.OfObject(Point, unlabeled).compare(new C.OfObject(Point, B.Point().x(1n).create())) === 0);
assert(new C.OfObject(Point).compare(new C.OfObject(Point)) === 0);
assert(new C.OfObject(Point, p).compare(new C.OfObject(Label, B.Label().name("a").create())) === null); // other schemas

## CMP-06 · Adjacencies do not take part in comparing objects

In [ ]:
const [t1, t2] = [B.Label().name("t").create(), B.Label().name("t").create()];
const tagged = B.Point().x(1n).label("a").tags((x: any) => x.tag(t1).note("n")).create();
assert(new C.OfObject(Point, p).compare(new C.OfObject(Point, tagged)) === 0);

## CMP-07 · Links compare by the identity of their targets

In [ ]:
const link = new C.OfLink("tag");
assert(link.name() === "tag" && link.compare(new C.OfLink("tag")) === 0);
raises(ValueError, () => link.target(() => undefined), "link 'tag' is not set");
assert(link.set(t1) === link && link.compare(new C.OfLink("tag")) === null && new C.OfLink("tag").compare(link) === null);
assert(link.compare(new C.OfLink("tag").set(t1)) === 0);
assert(link.compare(new C.OfLink("tag").set(t2)) === null); // equal content, different objects
const seen: unknown[] = [];
assert(link.target((t) => seen.push(t)) === link && same(seen, [t1]));

## CMP-08 · Entries are equal when their links and properties are

In [ ]:
/** An entry seen from its `item`, linking `tag`. */
function entry(tag?: Visitable, values: Record<string, Native> = {}): C.OfEntry {
  const recorded = new C.OfEntry(Tag, "item");
  if (tag !== undefined) recorded.link("tag", (k) => k.set(tag));
  for (const [name, value] of Object.entries(values)) set_property(recorded, name, value);
  return recorded;
}

assert(entry(t1, { note: "n" }).compare(entry(t1, { note: "n" })) === 0);
assert(entry(t1, { note: "n" }).compare(entry(t2, { note: "n" })) === null);
assert(entry(t1, { note: "n" }).compare(entry(t1, { note: "m" })) === null);
assert(entry(t1).compare(entry(t1, { note: "n" })) === null && entry(t1).compare(entry(t1)) === 0);
assert(entry(t1).compare(entry()) === null);
assert(entry(t1).compare(new C.OfEntry(Tag, "tag")) === null); // seen from the other end
assert(new C.OfEntry(Tag).compare(new C.OfEntry(Tag)) === 0); // no link implied, none set
const Other = new S.OfRelation.Builder().links("item", "tag").create();
assert(new C.OfEntry(Other, "item").compare(new C.OfEntry(Tag, "item")) === null);
raises(KeyError, () => entry().link("item", () => undefined), "'item' is not a link this entry can set"); // the object's own link
raises(KeyError, () => entry(undefined, { nope: "x" }), "unknown property 'nope'");

## CMP-09 · An adjacency's entries form a set

In [ ]:
/** The `tags` adjacency recorded from `point`. */
function tags(point: Visitable): C.OfAdjacency {
  const found: C.OfAdjacency[] = [];
  new C.OfObject(Point, point).adjacency("tags", (a) => found.push(a as C.OfAdjacency));
  return found[0] as C.OfAdjacency;
}

const both = B.Point().x(1n).tags((x: any) => x.tag(t1).note("n")).tags((x: any) => x.tag(t2)).create();
const reversed_ = B.Point().x(1n).tags((x: any) => x.tag(t2)).tags((x: any) => x.tag(t1).note("n")).create();
const one = B.Point().x(1n).tags((x: any) => x.tag(t1).note("n")).create();
const different = B.Point().x(1n).tags((x: any) => x.tag(t1).note("n")).tags((x: any) => x.tag(t1)).create();
assert(tags(both).compare(tags(reversed_)) === 0); // order does not matter
assert(tags(both).compare(tags(one)) === null && tags(both).compare(tags(different)) === null);
assert(tags(both).name() === "tags" && tags(both).me() === "item");
const fromLabel: C.OfAdjacency[] = [];
new C.OfObject(Label, t1).adjacency("items", (a) => fromLabel.push(a as C.OfAdjacency));
assert(tags(both).compare(fromLabel[0] as C.OfAdjacency) === null); // the same relation, seen from the other end
const Elsewhere = new S.OfObject.Builder().relations((r) => r.name("tags").of(Other).me("item")).create();
const elsewhere: C.OfAdjacency[] = [];
new C.OfObject(Elsewhere).adjacency("tags", (a) => elsewhere.push(a as C.OfAdjacency));
assert(tags(one).compare(elsewhere[0] as C.OfAdjacency) === null); // another relation

const adjacency = tags(one);
assert(adjacency.add((e) => { e.link("tag", (k) => k.set(t1)); set_property(e, "note", "n"); }) === adjacency);
let recorded: unknown[] = [];
assert(adjacency.entries((e) => recorded.push(e)) === adjacency && recorded.length === 1); // the equal entry was elided
adjacency.add((e) => e.link("tag", (k) => k.set(t2)));
recorded = [];
adjacency.entries((e) => recorded.push(e));
assert(recorded.length === 2 && adjacency.remove(recorded[0] as C.OfEntry) === adjacency);
recorded = [];
adjacency.entries((e) => recorded.push(e));
assert(recorded.length === 1);

## CMP-10 · Object-valued properties, and values of different kinds

In [ ]:
const Inner = new S.OfObject.Builder().ref().properties(text("x", BigInt)).create();
const Box = new S.OfObject.Builder().properties((p) => p.name("corner").of((t) => t.as_object(Inner))).create();

/** A box whose corner has `x`, written through the visitor protocols (proxies hold only natives). */
function box(x: bigint): C.OfObject {
  const recorded = new C.OfObject(Box);
  recorded.property("corner", (p) => p.value((a) => a.as_object((o) => set_property(o, "x", x))));
  return recorded;
}

assert(box(1n).compare(box(1n)) === 0 && box(1n).compare(box(2n)) === null && box(1n).compare(new C.OfObject(Box)) === null);
const reused = box(1n);
reused.property("corner", (p) => p.value((a) => a.as_object((o) => set_property(o, "x", 2n))));
assert(reused.compare(box(2n)) === 0); // the same recorded object, written twice
raises(TypeError, () => set_property(new C.OfObject(Box), "corner", 1n), "the schema is not a native schema");
raises(TypeError, () => new C.OfObject(Point).property("x", (p) => p.value((a) => a.as_object(() => undefined))),
  "the schema is not an object schema");
const native = new C.OfProperty("v", Int).value((a) => a.as_native((n) => n.set(1n)));
const obj = new C.OfProperty("v", Inner).value((a) => a.as_object(() => undefined));
assert(native.compare(obj) === null && obj.compare(native) === null);

## CMP-11 · Recordings read back through the visitor protocols

In [ ]:
const record = new C.OfObject(Point, p);
const names: unknown[] = [];
assert(record.properties((q) => names.push([q.name(), q.has()])) === record);
assert(JSON.stringify(names) === JSON.stringify([["x", true], ["label", true]]));
assert(record.has("x") && !new C.OfObject(Point).has("x") && get_property(record, "label") === "a");
assert(record.clear("label") === record && !record.has("label"));
record.property("x", (q) => q.clear());
names.length = 0;
record.properties((q) => names.push(q));
assert(!record.has("x") && names.length === 0);
seen.length = 0;
assert(record.adjacencies((a) => seen.push(a.name())) === record && same(seen, ["tags"]));
raises(KeyError, () => record.adjacency("nope", () => undefined), "unknown adjacency 'nope'");
raises(KeyError, () => record.property("nope", () => undefined), "unknown property 'nope'");
const written = new C.OfEntry(Tag, "item");
seen.length = 0;
assert(written.links((k) => { seen.push(k.name()); k.set(t1); }) === written);
set_property(written, "note", "n");
assert(written.has("note") && written.properties((q) => seen.push(q.name())) === written);
assert(same(seen, ["tag", "note"]) && written.clear("note") === written && !written.has("note"));

## CMP-12 · A union or intersection value needs a schema of its kind

In [ ]:
raises(TypeError, () => new C.OfAny(Int).as_intersection(() => undefined), "the schema is not an intersection schema");
raises(TypeError, () => new C.OfAny(Int).as_union(() => undefined), "the schema is not a union schema");